In [1]:
import pandas as pd
from pathlib import Path

data_path = Path("player_match_stats_v3.xlsx")
if data_path.exists():
    stats = pd.read_excel(data_path)
else:
    stats = pd.read_csv("player_match_stats_v3.csv")
df = stats.copy()

features = df.columns.tolist()[6:]

In [2]:
# Using group shuffle split as the data is imbalanced
from sklearn.model_selection import GroupShuffleSplit

X = df[features].copy()
y = df["MVP"]
groups = df["match_id"]

X["pass_accuracy"] = X["pass_accuracy"].fillna(0)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

In [3]:
# Logistic Regression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [4]:
# Classification Metrics
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score
)

y_prob = logistic_model.predict_proba(X_test)[:, 1]

print("ROC-AUC:",
      roc_auc_score(y_test, y_prob))

print("PR-AUC:",
      average_precision_score(y_test, y_prob))

print(
    classification_report(
        y_test,
        logistic_model.predict(X_test)
    )
)

ROC-AUC: 0.8318515655199072
PR-AUC: 0.35959978246834323
              precision    recall  f1-score   support

           0       0.99      0.89      0.94       398
           1       0.16      0.62      0.25        13

    accuracy                           0.89       411
   macro avg       0.57      0.75      0.60       411
weighted avg       0.96      0.89      0.92       411



In [5]:
# Top K Accuracy
test_results = df.iloc[test_idx].copy()

test_results["MVP_probability"] = y_prob

def top_k_accuracy(results, k=3):
    ranked = (
        results
        .sort_values(
            ["match_id", "MVP_probability"],
            ascending=[True, False]
        )
        .groupby("match_id")
        .head(k)
    )

    return (
        ranked
        .groupby("match_id")["MVP"]
        .max()
        .mean()
    )

print("Top-1:", top_k_accuracy(test_results, 1))
print("Top-3:", top_k_accuracy(test_results, 3))
print("Top-5:", top_k_accuracy(test_results, 5))

Top-1: 0.3076923076923077
Top-3: 0.46153846153846156
Top-5: 0.6153846153846154


In [6]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_prob = rf_model.predict_proba(X_test)[:, 1]

print("ROC-AUC:",
      roc_auc_score(y_test, y_prob))

print("PR-AUC:",
      average_precision_score(y_test, y_prob))

print(
    classification_report(
        y_test,
        rf_model.predict(X_test)
    )
)

test_results = df.iloc[test_idx].copy()

test_results["MVP_probability"] = y_prob

print("Top-1:", top_k_accuracy(test_results, 1))
print("Top-3:", top_k_accuracy(test_results, 3))
print("Top-5:", top_k_accuracy(test_results, 5))

ROC-AUC: 0.8662543486664089
PR-AUC: 0.24241912313831243
              precision    recall  f1-score   support

           0       0.98      0.96      0.97       398
           1       0.27      0.46      0.34        13

    accuracy                           0.94       411
   macro avg       0.63      0.71      0.66       411
weighted avg       0.96      0.94      0.95       411

Top-1: 0.38461538461538464
Top-3: 0.5384615384615384
Top-5: 0.6153846153846154


In [7]:
# XG Boost
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_prob = xgb_model.predict_proba(X_test)[:, 1]

print("ROC-AUC:",
      roc_auc_score(y_test, y_prob))

print("PR-AUC:",
      average_precision_score(y_test, y_prob))

print(
    classification_report(
        y_test,
        xgb_model.predict(X_test)
    )
)

test_results = df.iloc[test_idx].copy()

test_results["MVP_probability"] = y_prob

print("Top-1:", top_k_accuracy(test_results, 1))
print("Top-3:", top_k_accuracy(test_results, 3))
print("Top-5:", top_k_accuracy(test_results, 5))

ROC-AUC: 0.8419018167761886
PR-AUC: 0.4208641239266769
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       398
           1       0.60      0.23      0.33        13

    accuracy                           0.97       411
   macro avg       0.79      0.61      0.66       411
weighted avg       0.96      0.97      0.96       411

Top-1: 0.38461538461538464
Top-3: 0.5384615384615384
Top-5: 0.6923076923076923


In [8]:
# Deep Learning
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

class MVPNet(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)

model = MVPNet(input_dim=len(features))

#print(model)

num_negative = (y_train.to_numpy() == 0).sum()
num_positive = (y_train.to_numpy() == 1).sum()

pos_weight = num_negative / num_positive

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([pos_weight], dtype=torch.float32)
)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

num_epochs = 100

train_losses = []

for epoch in range(num_epochs):

    model.train()

    epoch_loss = 0.0

    for X_batch, y_batch in train_loader:

        # Forward pass
        logits = model(X_batch)

        # Calculate loss
        loss = criterion(logits, y_batch)

        # Clear previous gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        epoch_loss += loss.item() * len(X_batch)

    epoch_loss /= len(train_dataset)

    train_losses.append(epoch_loss)

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1:3d} | "
            f"Loss: {epoch_loss:.4f}"
        )

model.eval()

with torch.no_grad():

    test_logits = model(X_test_tensor)

    y_prob = torch.sigmoid(
        test_logits
    ).numpy().ravel()

print("ROC-AUC:",
      roc_auc_score(y_test, y_prob))

print("PR-AUC:",
      average_precision_score(y_test, y_prob))

print(
    classification_report(
        y_test,
        (y_prob >= 0.5).astype(int)
    )
)

test_results = df.iloc[test_idx].copy()

test_results["MVP_probability"] = y_prob

print("Top-1:", top_k_accuracy(test_results, 1))
print("Top-3:", top_k_accuracy(test_results, 3))
print("Top-5:", top_k_accuracy(test_results, 5))

Epoch  10 | Loss: 0.5716
Epoch  20 | Loss: 0.4289
Epoch  30 | Loss: 0.3435
Epoch  40 | Loss: 0.2959
Epoch  50 | Loss: 0.2612
Epoch  60 | Loss: 0.2277
Epoch  70 | Loss: 0.2222
Epoch  80 | Loss: 0.2135
Epoch  90 | Loss: 0.2047
Epoch 100 | Loss: 0.1333
ROC-AUC: 0.7962891379976808
PR-AUC: 0.1979622098531398
              precision    recall  f1-score   support

           0       0.98      0.95      0.96       398
           1       0.20      0.38      0.26        13

    accuracy                           0.93       411
   macro avg       0.59      0.67      0.61       411
weighted avg       0.95      0.93      0.94       411

Top-1: 0.38461538461538464
Top-3: 0.46153846153846156
Top-5: 0.6153846153846154


In [9]:
# Repeated grouped cross-validation
# Every fold keeps all rows from one match together.
import numpy as np
from sklearn.base import clone
from sklearn.metrics import average_precision_score, roc_auc_score

def repeated_group_folds(groups, n_splits=5, n_repeats=5, random_state=42):
    group_values = groups.to_numpy()
    unique_groups = np.unique(group_values)
    rng = np.random.RandomState(random_state)

    for repeat in range(n_repeats):
        shuffled_groups = rng.permutation(unique_groups)
        for test_groups in np.array_split(shuffled_groups, n_splits):
            test_mask = np.isin(group_values, test_groups)
            train_idx = np.flatnonzero(~test_mask)
            test_idx = np.flatnonzero(test_mask)
            yield repeat, train_idx, test_idx

def grouped_top_k(y_true, probabilities, match_ids, k):
    result = pd.DataFrame({
        "match_id": np.asarray(match_ids),
        "y": np.asarray(y_true),
        "p": np.asarray(probabilities)
    })
    selected = (
        result.sort_values(["match_id", "p"], ascending=[True, False])
        .groupby("match_id", sort=False)
        .head(k)
    )
    return selected.groupby("match_id")["y"].max().mean()

def evaluate_grouped_predictions(y_true, probabilities, match_ids):
    return {
        "PR-AUC": average_precision_score(y_true, probabilities),
        "ROC-AUC": roc_auc_score(y_true, probabilities),
        "Top-1": grouped_top_k(y_true, probabilities, match_ids, 1),
        "Top-3": grouped_top_k(y_true, probabilities, match_ids, 3),
        "Top-5": grouped_top_k(y_true, probabilities, match_ids, 5)
    }

model_templates = {
    "Logistic Regression": logistic_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

cv_rows = []
for model_name, model_template in model_templates.items():
    predictions = np.zeros(len(df))
    repeat_rows = []
    current_repeat = None

    for repeat, train_idx_cv, test_idx_cv in repeated_group_folds(
        groups, n_splits=5, n_repeats=5, random_state=42
    ):
        if current_repeat is not None and repeat != current_repeat:
            repeat_rows.append(evaluate_grouped_predictions(y, predictions, groups))
        current_repeat = repeat
        model = clone(model_template)
        model.fit(X.iloc[train_idx_cv], y.iloc[train_idx_cv])
        predictions[test_idx_cv] = model.predict_proba(X.iloc[test_idx_cv])[:, 1]

    repeat_rows.append(evaluate_grouped_predictions(y, predictions, groups))
    repeat_results = pd.DataFrame(repeat_rows)
    cv_rows.append({
        "Model": model_name,
        **{f"{column} mean": repeat_results[column].mean() for column in repeat_results},
        **{f"{column} std": repeat_results[column].std(ddof=1) for column in repeat_results}
    })

# Dynamic scoring baseline: equal weight for per-match and global ranks.
# Global percentiles are fitted on the training fold only.
positive_stats_base = [
    "passes", "shots", "carries", "pressures",
    "ball_recoveries", "duels", "clearances", "blocks",
    "dribbles", "interceptions", "fouls_won", "pass_accuracy",
    "goals", "shots_on_target", "xG"
]
negative_stats_base = ["fouls_committed", "dispossessed", "miscontrols"]
rank_stats = [
    column for column in positive_stats_base + negative_stats_base
    if column in X.columns
]
negative_stats = set(negative_stats_base)

def empirical_percentile_from_train(train_values, test_values):
    sorted_train = np.sort(np.asarray(train_values, dtype=float))
    left = np.searchsorted(sorted_train, test_values, side="left")
    right = np.searchsorted(sorted_train, test_values, side="right")
    # Midrank handling for ties, consistent with pandas rank(method="average").
    return (left + right) / (2.0 * len(sorted_train))

def dynamic_baseline_score(X_train_fold, X_test_fold, test_match_ids):
    per_match_ranks = []
    global_ranks = []
    test_match_ids = np.asarray(test_match_ids)

    for column in rank_stats:
        test_values = X_test_fold[column].to_numpy(dtype=float)
        per_match = (
            pd.Series(test_values)
            .groupby(test_match_ids)
            .rank(pct=True)
            .to_numpy()
        )
        global_rank = empirical_percentile_from_train(
            X_train_fold[column].to_numpy(dtype=float), test_values
        )

        if column in negative_stats:
            per_match = 1.0 - per_match
            global_rank = 1.0 - global_rank

        per_match_ranks.append(per_match)
        global_ranks.append(global_rank)

    per_match_score = np.column_stack(per_match_ranks).mean(axis=1)
    global_score = np.column_stack(global_ranks).mean(axis=1)
    return 0.5 * per_match_score + 0.5 * global_score

baseline_repeat_rows = []
baseline_predictions = np.zeros(len(df))
current_repeat = None
for repeat, train_idx_cv, test_idx_cv in repeated_group_folds(
    groups, n_splits=5, n_repeats=5, random_state=42
):
    if current_repeat is not None and repeat != current_repeat:
        baseline_repeat_rows.append(
            evaluate_grouped_predictions(y, baseline_predictions, groups)
        )
    current_repeat = repeat
    baseline_predictions[test_idx_cv] = dynamic_baseline_score(
        X.iloc[train_idx_cv],
        X.iloc[test_idx_cv],
        groups.iloc[test_idx_cv]
    )
baseline_repeat_rows.append(
    evaluate_grouped_predictions(y, baseline_predictions, groups)
)
baseline_results = pd.DataFrame(baseline_repeat_rows)
cv_rows.append({
    "Model": "Dynamic Score (0.5/0.5)",
    **{f"{column} mean": baseline_results[column].mean() for column in baseline_results},
    **{f"{column} std": baseline_results[column].std(ddof=1) for column in baseline_results}
})

# Neural network: train a fresh network and scaler inside every fold.
def train_nn_predict_proba(X_train_fold, y_train_fold, X_test_fold, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

    fold_scaler = StandardScaler()
    X_train_scaled = fold_scaler.fit_transform(X_train_fold)
    X_test_scaled = fold_scaler.transform(X_test_fold)

    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(
        y_train_fold.to_numpy(), dtype=torch.float32
    ).reshape(-1, 1)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

    fold_model = MVPNet(input_dim=X_train_fold.shape[1])
    num_negative = (y_train_fold.to_numpy() == 0).sum()
    num_positive = (y_train_fold.to_numpy() == 1).sum()
    fold_pos_weight = num_negative / num_positive
    fold_criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([fold_pos_weight], dtype=torch.float32)
    )
    fold_optimizer = torch.optim.Adam(
        fold_model.parameters(), lr=0.001, weight_decay=1e-4
    )

    fold_model.train()
    for _ in range(150):
        logits = fold_model(X_train_tensor)
        loss = fold_criterion(logits, y_train_tensor)
        fold_optimizer.zero_grad()
        loss.backward()
        fold_optimizer.step()

    fold_model.eval()
    with torch.no_grad():
        return torch.sigmoid(fold_model(X_test_tensor)).numpy().ravel()

nn_repeat_rows = []
nn_predictions = np.zeros(len(df))
current_repeat = None
for repeat, train_idx_cv, test_idx_cv in repeated_group_folds(
    groups, n_splits=5, n_repeats=5, random_state=42
):
    if current_repeat is not None and repeat != current_repeat:
        nn_repeat_rows.append(
            evaluate_grouped_predictions(y, nn_predictions, groups)
        )
    current_repeat = repeat
    nn_predictions[test_idx_cv] = train_nn_predict_proba(
        X.iloc[train_idx_cv], y.iloc[train_idx_cv], X.iloc[test_idx_cv],
        seed=42 + repeat
    )
nn_repeat_rows.append(evaluate_grouped_predictions(y, nn_predictions, groups))

nn_results = pd.DataFrame(nn_repeat_rows)
cv_rows.append({
    "Model": "Neural Network",
    **{f"{column} mean": nn_results[column].mean() for column in nn_results},
    **{f"{column} std": nn_results[column].std(ddof=1) for column in nn_results}
})

cv_summary = pd.DataFrame(cv_rows).set_index("Model")
display(cv_summary.round(3))

,PR-AUC mean,ROC-AUC mean,Top-1 mean,Top-3 mean,Top-5 mean,PR-AUC std,ROC-AUC std,Top-1 std,Top-3 std,Top-5 std
Model,,,,,,,,,,
Logistic Regression,0.364,0.888,0.425,0.681,0.791,0.011,0.007,0.034,0.018,0.024
Random Forest,0.291,0.898,0.428,0.672,0.784,0.012,0.009,0.042,0.025,0.037
XGBoost,0.331,0.900,0.434,0.706,0.816,0.015,0.007,0.056,0.028,0.017
Dynamic Score (0.5/0.5),0.102,0.714,0.106,0.297,0.428,0.001,0.000,0.007,0.000,0.009
Neural Network,0.358,0.881,0.453,0.688,0.781,0.014,0.008,0.033,0.025,0.041


In [16]:
# Seven-way comparison: classifier, relative features, winner reranking, and XGBRanker
# All variants use the same repeated grouped folds for a fair comparison.
from xgboost import XGBClassifier, XGBRanker

N_SPLITS = 5
N_REPEATS = 5

def build_relative_features(data, raw_features):
    relative = pd.DataFrame(index=data.index)
    match_ids = data["match_id"]
    team_keys = [data["match_id"], data["team"]]

    for column in raw_features:
        values = data[column].astype(float)

        relative[f"{column}__match_pct"] = (
            values.groupby(match_ids).rank(pct=True)
        )

        match_total = values.groupby(match_ids).transform("sum")
        relative[f"{column}__match_share"] = (
            values.div(match_total.replace(0, np.nan)).fillna(0)
        )

        relative[f"{column}__team_pct"] = (
            values.groupby(team_keys).rank(pct=True)
        )

        team_total = values.groupby(team_keys).transform("sum")
        relative[f"{column}__team_share"] = (
            values.div(team_total.replace(0, np.nan)).fillna(0)
        )

        team_mean = values.groupby(team_keys).transform("mean")
        relative[f"{column}__minus_team_mean"] = values - team_mean

    return relative.replace([np.inf, -np.inf], 0).fillna(0)

relative_features = build_relative_features(df, X.columns.tolist())
X_relative = pd.concat([X, relative_features], axis=1)

# Match-result information is used only for post-prediction reranking.
team_goals = df.groupby(["match_id", "team"])["goals"].transform("sum")
max_match_goals = team_goals.groupby(df["match_id"]).transform("max")
winner_eligible = (team_goals == max_match_goals).to_numpy()

def apply_winner_only_reranking(scores, eligible_mask):
    scores = np.asarray(scores, dtype=float).copy()
    if np.any(~eligible_mask):
        finite_scores = scores[np.isfinite(scores)]
        floor = finite_scores.min() - 1.0
        scores[~eligible_mask] = floor
    return scores

def make_xgb_classifier():
    return XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

def make_xgb_ranker():
    return XGBRanker(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="rank:pairwise",
        eval_metric="ndcg@5",
        random_state=42,
        n_jobs=-1
    )

def fit_ranker_and_predict(ranker, X_train_fold, y_train_fold, groups_train_fold, X_test_fold):
    train_groups = groups_train_fold.to_numpy()
    train_order = np.argsort(train_groups, kind="stable")
    X_train_sorted = X_train_fold.iloc[train_order]
    y_train_sorted = y_train_fold.iloc[train_order]
    groups_sorted = train_groups[train_order]

    try:
        ranker.fit(
            X_train_sorted, y_train_sorted, qid=groups_sorted
        )
    except TypeError:
        group_sizes = (
            pd.Series(groups_sorted)
            .groupby(groups_sorted, sort=False)
            .size()
            .to_numpy()
        )
        ranker.fit(
            X_train_sorted, y_train_sorted, group=group_sizes
        )

    return ranker.predict(X_test_fold)

variant_prediction_store = {}

def evaluate_variant(model_name, feature_matrix, model_kind, winner_only=False):
    repeat_rows = []
    predictions = np.zeros(len(df))
    current_repeat = None

    for repeat, train_idx_cv, test_idx_cv in repeated_group_folds(
        groups, n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=42
    ):
        if current_repeat is not None and repeat != current_repeat:
            repeat_rows.append(
                evaluate_grouped_predictions(y, predictions, groups)
            )
        current_repeat = repeat

        X_train_fold = feature_matrix.iloc[train_idx_cv]
        X_test_fold = feature_matrix.iloc[test_idx_cv]
        y_train_fold = y.iloc[train_idx_cv]
        groups_train_fold = groups.iloc[train_idx_cv]

        if model_kind == "classifier":
            model = make_xgb_classifier()
            model.fit(X_train_fold, y_train_fold)
            fold_scores = model.predict_proba(X_test_fold)[:, 1]
        else:
            model = make_xgb_ranker()
            fold_scores = fit_ranker_and_predict(
                model, X_train_fold, y_train_fold, groups_train_fold, X_test_fold
            )

        if winner_only:
            fold_scores = apply_winner_only_reranking(
                fold_scores, winner_eligible[test_idx_cv]
            )
        predictions[test_idx_cv] = fold_scores

    repeat_rows.append(
        evaluate_grouped_predictions(y, predictions, groups)
    )
    variant_prediction_store[model_name] = predictions.copy()
    result = pd.DataFrame(repeat_rows)
    return {
        "Model": model_name,
        **{f"{column} mean": result[column].mean() for column in result},
        **{f"{column} std": result[column].std(ddof=1) for column in result}
    }

seven_variant_rows = [
    evaluate_variant(
        "Base XGBoost Classifier", X, "classifier", winner_only=False
    ),
    evaluate_variant(
        "Base XGBoost + Relative Features",
        X_relative, "classifier", winner_only=False
    ),
    evaluate_variant(
        "Base XGBoost + Winner-only Reranking",
        X, "classifier", winner_only=True
    ),
    evaluate_variant(
        "XGBRanker + Raw Features", X, "ranker", winner_only=False
    ),
    evaluate_variant(
        "XGBRanker + Relative Features",
        X_relative, "ranker", winner_only=False
    ),
    evaluate_variant(
        "XGBRanker + Winner-only Reranking",
        X, "ranker", winner_only=True
    ),
    evaluate_variant(
        "XGBRanker + Relative Features + Winner-only Reranking",
        X_relative, "ranker", winner_only=True
    )
]

seven_variant_summary = pd.DataFrame(seven_variant_rows).set_index("Model")
display(seven_variant_summary.round(3))

,PR-AUC mean,ROC-AUC mean,Top-1 mean,Top-3 mean,Top-5 mean,PR-AUC std,ROC-AUC std,Top-1 std,Top-3 std,Top-5 std
Model,,,,,,,,,,
Base XGBoost Classifier,0.331,0.900,0.434,0.706,0.816,0.015,0.007,0.056,0.028,0.017
Base XGBoost + Relative Features,0.369,0.920,0.494,0.734,0.872,0.020,0.006,0.042,0.011,0.028
Base XGBoost + Winner-only Reranking,0.397,0.937,0.503,0.806,0.872,0.015,0.004,0.034,0.014,0.020
XGBRanker + Raw Features,0.339,0.897,0.447,0.656,0.775,0.008,0.008,0.014,0.044,0.024
XGBRanker + Relative Features,0.345,0.917,0.466,0.712,0.869,0.021,0.007,0.026,0.026,0.032
XGBRanker + Winner-only Reranking,0.402,0.936,0.506,0.769,0.875,0.006,0.004,0.021,0.013,0.016
XGBRanker + Relative Features + Winner-only Reranking,0.402,0.945,0.528,0.828,0.903,0.016,0.004,0.030,0.037,0.007


In [ ]:
performance_model_key = "XGBRanker + Relative Features"
if performance_model_key not in variant_prediction_store:
    raise RuntimeError("Run last cell for comparison first.")

performance_rank_score = variant_prediction_store[performance_model_key]
winner_only_rank_score = apply_winner_only_reranking(
    performance_rank_score, winner_eligible
)
team_goals_export = df.groupby(["match_id", "team"])["goals"].transform("sum")
match_goals_export = df.groupby("match_id")["goals"].transform("sum")
opponent_goals_export = match_goals_export - team_goals_export
match_outcome_export = np.select(
    [team_goals_export > opponent_goals_export, team_goals_export < opponent_goals_export],
    ["W", "L"], default="D"
)

export_df = df.copy()
export_df["team_goals"] = team_goals_export.to_numpy()
export_df["opponent_goals"] = opponent_goals_export.to_numpy()
export_df["match_outcome"] = match_outcome_export
export_df["winner_eligible"] = winner_eligible.astype(int)
export_df["model_name"] = performance_model_key + " (OOF; winner-only MVP selection)"
# This is the player-performance score and is not winner-masked.
export_df["model_rank_score"] = performance_rank_score
export_df["MVP_score"] = export_df.groupby("match_id")["model_rank_score"].rank(pct=True) * 100
export_df["performance_rank"] = export_df.groupby("match_id")["model_rank_score"].rank(ascending=False, method="min")
# Winner-only is applied only to the final MVP selection rank.
export_df["winner_only_rank_score"] = winner_only_rank_score
export_df["MVP_rank"] = export_df.groupby("match_id")["winner_only_rank_score"].rank(ascending=False, method="min")
export_df["model_top1_candidate"] = (export_df["MVP_rank"] == 1).astype(int)
export_df["oof_score_tie_count"] = export_df.groupby(["match_id", "model_rank_score"])["player_id"].transform("size")
export_df["dynamic_score_0_100"] = dynamic_baseline_score(X, X, groups) * 100

export_df.to_csv("player_performance_rank.csv", index=False, encoding="utf-8-sig")
display(export_df[["match_id", "player_id", "player", "team", "MVP", "match_outcome", "winner_eligible", "model_rank_score", "MVP_score", "performance_rank", "winner_only_rank_score", "MVP_rank", "oof_score_tie_count"]].sort_values(["match_id", "MVP_rank"]).head(20))
print("Saved file: player_performance_rank.csv")

,match_id,player_id,player,team,MVP,match_outcome,winner_eligible,model_rank_score,MVP_score,MVP_rank,oof_score_tie_count
28,3857254,105943,Aymen Dahmen,Tunisia,0,D,1,0.435861,100.000000,1.0,1
10,3857254,5655,Dylan Daniel Mahmoud Bronn,Tunisia,0,D,1,-0.333847,96.774194,2.0,1
20,3857254,23910,Youssef Msakni,Tunisia,0,D,1,-0.708659,93.548387,3.0,1
0,3857254,3043,Christian Dannemann Eriksen,Denmark,0,D,1,-0.790696,90.322581,4.0,1
11,3857254,5732,Andreas Evald Cornelius,Denmark,0,D,1,-1.150756,87.096774,5.0,1
23,3857254,30681,Ali Abdi,Tunisia,0,D,1,-1.654949,83.870968,6.0,1
13,3857254,8247,Joachim Andersen,Denmark,0,D,1,-1.736633,80.645161,7.0,1
1,3857254,3557,Naïm Sliti,Tunisia,0,D,1,-2.041618,77.419355,8.0,1
26,3857254,44955,Aïssa Bilal Laïdouni,Tunisia,1,D,1,-2.067955,74.193548,9.0,1
3,3857254,3767,Ellyes Joris Skhiri,Tunisia,0,D,1,-2.121896,70.967742,10.0,1


Saved OOF Power BI export: player_performance_rank.csv
